# Cardiac Risk Stratification System: TinyML & ECG/PPG Telemetry Pipeline

This notebook contains the complete end-to-end code for the wearable cardiac risk stratification model combining **ECG and PPG telemetry with TinyML**.

### Key Architecture Included:
1. **Data Preprocessing & Feature Extraction** (MIT-BIH Arrhythmia telemetry simulation)
2. **Class Balancing with SMOTE**
3. **Random Forest Classifier Training & Evaluation (Macro F1-Score)**
4. **TinyML Model Export (C/C++ rules for ESP32 microcontroller)**
5. **PythonAnywhere Telemetry Backend API Script**

## 1. Environment & Dependencies

In [ ]:
# Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)

## 2. Dataset Preprocessing & Feature Engineering

In [ ]:
# Synthetic telemetry feature loader representing MIT-BIH dataset parameters
def load_cardiac_dataset(n_samples=2000):
    np.random.seed(SEED)
    # Features: [rr_mean, rr_std, qrs_duration, p_amp, spo2, bpm]
    X_normal = np.random.normal(loc=[0.8, 0.05, 0.08, 0.12, 98.0, 72.0], scale=[0.05, 0.01, 0.005, 0.01, 1.0, 5.0], size=(n_samples // 2, 6))
    X_arrhythmia = np.random.normal(loc=[0.6, 0.15, 0.14, 0.05, 94.0, 110.0], scale=[0.1, 0.04, 0.02, 0.03, 2.5, 15.0], size=(n_samples // 4, 6))
    X_risk = np.random.normal(loc=[0.45, 0.22, 0.18, 0.02, 89.0, 135.0], scale=[0.12, 0.06, 0.03, 0.02, 3.0, 20.0], size=(n_samples // 4, 6))
    
    X = np.vstack([X_normal, X_arrhythmia, X_risk])
    # Labels: 0 = Normal, 1 = Moderate Risk (Arrhythmia), 2 = High Risk
    y = np.array([0]*(n_samples // 2) + [1]*(n_samples // 4) + [2]*(n_samples // 4))
    
    feature_names = ['rr_mean', 'rr_std', 'qrs_duration', 'p_amp', 'spo2', 'bpm']
    return pd.DataFrame(X, columns=feature_names), y

df_features, labels = load_cardiac_dataset()
print('Dataset Shape:', df_features.shape)
print('Class Distribution:', pd.Series(labels).value_counts().to_dict())
df_features.head()

## 3. Train-Test Split & SMOTE Balancing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_features, labels, test_size=0.2, random_state=SEED, stratify=labels
)

# Apply SMOTE oversampling to rectify imbalance
smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Before SMOTE distribution:', pd.Series(y_train).value_counts().to_dict())
print('After SMOTE distribution:', pd.Series(y_train_res).value_counts().to_dict())

## 4. Random Forest Classifier Training

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=SEED,
    n_jobs=-1
)

rf_clf.fit(X_train_res, y_train_res)
print('Model successfully trained!')

## 5. Model Evaluation (Macro F1 & Confusion Matrix)

In [ ]:
y_pred = rf_clf.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average='macro')
acc = accuracy_score(y_test, y_pred)

print(f'Overall Accuracy: {acc * 100:.2f}%')
print(f'Macro F1-Score:   {macro_f1:.4f}\n')
print('Classification Report:\n', classification_report(y_test, y_pred, target_names=['Normal', 'Moderate Risk', 'High Risk']))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Mod Risk', 'High Risk'], yticklabels=['Normal', 'Mod Risk', 'High Risk'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Cardiac Risk Classification Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Model Export & Embedded ESP32 Microcontroller Conversion

In [ ]:
# Save Python trained model artifact
joblib.dump(rf_clf, 'cardiac_risk_rf_model.pkl')
print('Saved model to cardiac_risk_rf_model.pkl')

# C/C++ Header snippet generator for TinyML on ESP32 microcontrollers
c_header_content = '''
// Embedded C/C++ Inference Rule for ESP32 Microcontroller
#ifndef CARDIAC_RISK_MODEL_H
#define CARDIAC_RISK_MODEL_H

int classify_cardiac_risk(float rr_mean, float rr_std, float qrs_dur, float p_amp, float spo2, float bpm) {
    if (spo2 < 90.0f || bpm > 130.0f) {
        return 2; // High Risk
    } else if (rr_std > 0.15f || qrs_dur > 0.12f) {
        return 1; // Moderate Risk / Arrhythmia
    } else {
        return 0; // Normal
    }
}

#endif // CARDIAC_RISK_MODEL_H
'''

with open('cardiac_risk_tinyml.h', 'w') as f:
    f.write(c_header_content)
print('Generated TinyML C++ header file for ESP32 target: cardiac_risk_tinyml.h')

## 7. PythonAnywhere Cloud API Server Implementation

In [ ]:
# Exportable FastAPI Cloud Server Script
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Cardiac Telemetry Risk Assessment API")
model = joblib.load("cardiac_risk_rf_model.pkl")

class TelemetryPayload(BaseModel):
    device_id: str
    rr_mean: float
    rr_std: float
    qrs_duration: float
    p_amp: float
    spo2: float
    bpm: float

@app.post("/predict")
def predict_risk(payload: TelemetryPayload):
    features = np.array([[
        payload.rr_mean, payload.rr_std, payload.qrs_duration, 
        payload.p_amp, payload.spo2, payload.bpm
    ]])
    prediction = int(model.predict(features)[0])
    risk_labels = {0: "Normal", 1: "Moderate Risk", 2: "High Risk"}
    
    return {
        "device_id": payload.device_id,
        "risk_code": prediction,
        "status": risk_labels[prediction]
    }
'''

with open('pythonanywhere_server.py', 'w') as f:
    f.write(api_code)
print('Generated PythonAnywhere server integration file: pythonanywhere_server.py')